# AMFI Pruning: complete research notebook

Runs CIFAR-10 training and pruning with measured outputs. Default FAST mode is a pipeline check; set `MODE="FULL"` for multi-seed research. Test data is reserved for the final frozen evaluation.

## 1. Project introduction
This notebook contains the executable core inline so it can run independently. Research results are written to Drive and are never fabricated.

In [ ]:
MODE = "FAST"  # FAST or FULL
SEEDS = [42] if MODE == "FAST" else [17, 42, 73]
SPARSITIES = [0.2, 0.5] if MODE == "FAST" else [0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9]
EPOCHS = 1 if MODE == "FAST" else 20
CALIBRATION_BATCHES = 2 if MODE == "FAST" else 16
BATCH_SIZE = 128

## 2. Environment setup

In [ ]:
%pip -q install torch torchvision scikit-learn pandas matplotlib seaborn reportlab

## 3. GPU and CUDA detection

In [ ]:
import os, time, json, math, random, uuid, copy, platform, hashlib, zipfile, io, shutil
from pathlib import Path
import numpy as np, pandas as pd, torch, torchvision
from torch import nn
DEVICE=torch.device("cuda" if torch.cuda.is_available() else "cpu")
print({"python":platform.python_version(),"torch":torch.__version__,"torchvision":torchvision.__version__,"device":str(DEVICE),"gpu":torch.cuda.get_device_name(0) if torch.cuda.is_available() else None})

## 4. Google Drive persistence and resume

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
ROOT=Path("/content/drive/MyDrive/AMFI-Pruning"); ROOT.mkdir(parents=True,exist_ok=True)
RESULTS=ROOT/"experiment_results.csv"
completed=pd.read_csv(RESULTS) if RESULTS.exists() else pd.DataFrame()
print("Existing persisted rows:",len(completed))

In [ ]:
# Pre-download synthetic self-test: fail fast before CIFAR download or training.
class PreflightNet(nn.Module):
 def __init__(self):
  super().__init__(); self.conv=nn.Conv2d(3,4,3,padding=1); self.pool=nn.AdaptiveAvgPool2d(1); self.fc=nn.Linear(4,3); self.saved=[]
 def forward(self,x):
  self.saved.append(x.detach().abs().mean((0,2,3))); a=self.conv(x); return self.fc(self.pool(torch.relu(a)).flatten(1))
tiny=PreflightNet().to(DEVICE); xx=torch.randn(8,3,8,8,device=DEVICE); yy=torch.randint(0,3,(8,),device=DEVICE)
loss=nn.functional.cross_entropy(tiny(xx),yy); loss.backward(); mag=tiny.conv.weight.detach().abs(); grad=(tiny.conv.weight.detach()*tiny.conv.weight.grad.detach()).abs(); act=mag*tiny.saved[-1].reshape(1,-1,1,1); layer=(grad.mean()+act.mean())/2
assert torch.isfinite(mag).all() and torch.isfinite(grad).all() and torch.isfinite(act).all() and torch.isfinite(layer)
scores=mag/mag.max().clamp_min(1e-12)+grad/grad.max().clamp_min(1e-12)+act/act.max().clamp_min(1e-12); flat=scores.flatten(); dead=torch.zeros(flat.numel(),dtype=torch.bool,device=DEVICE); dead[torch.argsort(flat)[:round(.3*flat.numel())]]=True; mask=(~dead).reshape_as(scores); assert int((~mask).sum())==round(.3*flat.numel())
with torch.no_grad(): tiny.conv.weight.mul_(mask)
opt=torch.optim.AdamW(tiny.parameters(),weight_decay=.1)
for _ in range(2):
 opt.zero_grad(); nn.functional.cross_entropy(tiny(xx),yy).backward(); tiny.conv.weight.grad.mul_(mask); opt.step()
 with torch.no_grad(): tiny.conv.weight.mul_(mask)
assert torch.count_nonzero(tiny.conv.weight[~mask])==0 and tiny(xx).shape==(8,3)
torch.save(tiny.state_dict(),ROOT/"pre_download_selftest.pt"); state=torch.load(ROOT/"pre_download_selftest.pt",map_location=DEVICE,weights_only=True); tiny.load_state_dict(state); print("Pre-download synthetic AMFI/mask/model/checkpoint self-test passed")

## 5–8. CIFAR-10, stratified 45k/5k split, train-only augmentation

In [ ]:
from torchvision import datasets,transforms
from torch.utils.data import DataLoader,Subset
from sklearn.model_selection import train_test_split
root="/content/data"
train_aug=transforms.Compose([transforms.RandomCrop(32,padding=4),transforms.RandomHorizontalFlip(),transforms.ToTensor(),transforms.Normalize((.4914,.4822,.4465),(.247,.243,.261))])
eval_tf=transforms.Compose([transforms.ToTensor(),transforms.Normalize((.4914,.4822,.4465),(.247,.243,.261))])
raw=datasets.CIFAR10(root,train=True,download=True); tr,va=train_test_split(np.arange(len(raw)),test_size=5000,stratify=raw.targets,random_state=42)
trainset=datasets.CIFAR10(root,train=True,download=False,transform=train_aug); evaltrain=datasets.CIFAR10(root,train=True,download=False,transform=eval_tf); testset=datasets.CIFAR10(root,train=False,download=True,transform=eval_tf)
train_loader=DataLoader(Subset(trainset,tr),BATCH_SIZE,shuffle=True,num_workers=2); val_loader=DataLoader(Subset(evaltrain,va),BATCH_SIZE,shuffle=False,num_workers=2); test_loader=DataLoader(testset,BATCH_SIZE,shuffle=False,num_workers=2)
cal_loader=DataLoader(Subset(evaltrain,tr[:min(4000,len(tr))]),BATCH_SIZE,shuffle=False,num_workers=2)

## 9–12. CIFAR ResNet-18, baseline training, and inline AMFI core

In [ ]:
from torchvision.models import resnet18
from torch import nn
def make_model():
 m=resnet18(weights=None,num_classes=10); m.conv1=nn.Conv2d(3,64,3,1,1,bias=False); m.maxpool=nn.Identity(); return m
def seed_all(s):
 random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s) if torch.cuda.is_available() else None
def eval_acc(model,loader):
 model.eval(); ok=n=0
 with torch.inference_mode():
  for x,y in loader: x,y=x.to(DEVICE),y.to(DEVICE); ok+=(model(x).argmax(1)==y).sum().item(); n+=len(y)
 return ok/n
def train(model,loader,opt,epochs,masks=None):
 hist=[]; lossfn=nn.CrossEntropyLoss()
 for ep in range(epochs):
  model.train(); total=loss_sum=0
  for x,y in loader:
   x,y=x.to(DEVICE),y.to(DEVICE); opt.zero_grad(set_to_none=True); loss=lossfn(model(x),y); loss.backward()
   if masks:
    for n,p in model.named_parameters():
     if n in masks and p.grad is not None: p.grad.mul_(masks[n].to(DEVICE))
   opt.step()
   if masks:
    with torch.no_grad():
     for n,p in model.named_parameters():
      if n in masks: p.mul_(masks[n].to(DEVICE))
   total+=len(y); loss_sum+=loss.item()*len(y)
  hist.append({"epoch":ep+1,"loss":loss_sum/total,"val_accuracy":eval_acc(model,val_loader)}); print(hist[-1])
 return hist
def norm(x,mode="layer_minmax"):
 x=torch.nan_to_num(x.float());
 if mode=="layer_mean": return (x/x.mean().clamp_min(1e-12)).clamp(0,1)
 if mode=="percentile": return (x/x.flatten().quantile(.99).clamp_min(1e-12)).clamp(0,1)
 lo,hi=x.min(),x.max(); return (x-lo)/(hi-lo) if hi>lo else torch.zeros_like(x)
def importance(model,loader,batches=8):
 ps={n:p for n,p in model.named_parameters() if p.requires_grad and p.ndim>=2 and not n.endswith("bias")}; grad={n:torch.zeros_like(p,device="cpu") for n,p in ps.items()}; acts={n:[] for n in ps}; handles=[]; mods=dict(model.named_modules())
 for n in ps:
  mod=mods.get(n.rsplit(".",1)[0])
  if isinstance(mod,(nn.Conv2d,nn.Linear)):
   def hook(m,args,out,key=n):
    if args and torch.is_tensor(args[0]): acts[key].append(args[0].detach().abs().float().mean(tuple(i for i in range(args[0].ndim) if i!=1)).cpu())
   handles.append(mod.register_forward_hook(hook))
 model.eval()
 for i,(x,y) in enumerate(loader):
  if i>=batches: break
  x,y=x.to(DEVICE),y.to(DEVICE); model.zero_grad(set_to_none=True); nn.functional.cross_entropy(model(x),y).backward()
  for n,p in ps.items():
   if p.grad is not None: grad[n]+=(p.detach()*p.grad.detach()).abs().cpu()/batches
 for h in handles: h.remove()
 mag={n:p.detach().abs().cpu() for n,p in ps.items()}; act={}
 for n,p in ps.items():
  av=torch.stack(acts[n]).mean(0) if acts[n] else torch.zeros(p.shape[1]); shape=[1]*p.ndim; shape[1]=len(av); act[n]=mag[n]*av.reshape(shape)
 raw={n:(grad[n].mean()+act[n].mean())/2 for n in ps}; vals=norm(torch.stack(list(raw.values())),"global"); layer=dict(zip(raw,vals.tolist()))
 return {"magnitude":mag,"gradient":grad,"activation":act,"layer_importance":layer}
def amfi_scores(c,coeff=(.25,.35,.25,.15),mode="layer_minmax"):
 out={}; names=list(c["magnitude"])
 for n in names:
  a,b,g=[norm(c[k][n],mode) for k in ("magnitude","gradient","activation")]; out[n]=coeff[0]*a+coeff[1]*b+coeff[2]*g+coeff[3]*c["magnitude"][n].new_full(a.shape,c["layer_importance"][n])
 return out
def make_masks(scores,s):
 flat=torch.cat([v.flatten() for v in scores.values()]); k=round(s*flat.numel()); dead=torch.zeros(flat.numel(),dtype=torch.bool); dead[torch.argsort(flat,stable=True)[:k]]=True; out={}; off=0
 for n,v in scores.items(): out[n]=~dead[off:off+v.numel()].reshape(v.shape); off+=v.numel()
 return out

In [ ]:
# Override the initial helper with concatenated cross-layer scaling for global mode.
def amfi_scores(c,coeff=(.25,.35,.25,.15),mode="layer_minmax"):
 names=list(c["magnitude"]); normalized={}
 for key in ("magnitude","gradient","activation"):
  if mode=="global":
   flat=torch.cat([c[key][n].flatten() for n in names]); scaled=norm(flat,"global"); offset=0; normalized[key]={}
   for n in names:
    size=c[key][n].numel(); normalized[key][n]=scaled[offset:offset+size].reshape_as(c[key][n]); offset+=size
  else: normalized[key]={n:norm(c[key][n],mode) for n in names}
 out={}
 for n in names:
  a,b,g=[normalized[key][n] for key in ("magnitude","gradient","activation")]; layer=c["magnitude"][n].new_full(a.shape,c["layer_importance"][n]); out[n]=coeff[0]*a+coeff[1]*b+coeff[2]*g+coeff[3]*layer
 return out

## 13. Synthetic self-test (must pass before dataset training)

In [ ]:
tiny=nn.Sequential(nn.Conv2d(3,4,3,padding=1),nn.ReLU(),nn.Flatten(),nn.Linear(4*8*8,3)).to(DEVICE)
x=torch.randn(6,3,8,8,device=DEVICE); y=torch.randint(0,3,(6,),device=DEVICE); opt=torch.optim.AdamW(tiny.parameters(),weight_decay=.1)
components=importance(tiny,[(x,y)],batches=1); assert all(k in components for k in ("magnitude","gradient","activation","layer_importance"))
sc=amfi_scores(components); masks=make_masks(sc,.4); assert abs(sum((~m).sum().item() for m in masks.values())/sum(m.numel() for m in masks.values())-.4)<.01
for _ in range(2):
 opt.zero_grad(); nn.functional.cross_entropy(tiny(x),y).backward()
 for n,p in tiny.named_parameters():
  if n in masks: p.grad.mul_(masks[n].to(DEVICE))
 opt.step()
 with torch.no_grad():
  for n,p in tiny.named_parameters():
   if n in masks: p.mul_(masks[n].to(DEVICE))
assert all(torch.count_nonzero(dict(tiny.named_parameters())[n][~m]).item()==0 for n,m in masks.items())
torch.save(tiny.state_dict(),ROOT/"synthetic_selftest.pt"); tiny.load_state_dict(torch.load(ROOT/"synthetic_selftest.pt",map_location=DEVICE,weights_only=True)); print("SELF-TEST PASSED")

## 14–20. Train once per seed; baseline methods; fixed and adaptive AMFI

In [ ]:
import torch.nn.utils.prune as prune
COEFFICIENTS={"A":(.25,.35,.25,.15),"B":(.4,.3,.2,.1),"C":(.2,.4,.3,.1),"D":(.3,.25,.3,.15)}
def baseline_model(seed):
 seed_all(seed); m=make_model().to(DEVICE); opt=torch.optim.SGD(m.parameters(),lr=.1,momentum=.9,weight_decay=5e-4); hist=train(m,train_loader,opt,EPOCHS); torch.save({"model":m.state_dict(),"optimizer":opt.state_dict(),"history":hist,"seed":seed},ROOT/f"baseline_{MODE}_{seed}.pt"); return m
def prune_method(base,method,sparsity,seed,coeff=(.25,.35,.25,.15)):
 m=copy.deepcopy(base).to(DEVICE); ps={n:p for n,p in m.named_parameters() if p.ndim>=2 and not n.endswith("bias")}
 if method=="amfi": scores=amfi_scores(importance(m,cal_loader,CALIBRATION_BATCHES),coeff); masks=make_masks(scores,sparsity)
 elif method in ("magnitude","random"): scores={n:(p.detach().abs().cpu() if method=="magnitude" else torch.rand(p.shape,generator=torch.Generator().manual_seed(seed))) for n,p in ps.items()}; masks=make_masks(scores,sparsity)
 elif method=="l1":
  modules=[mod for mod in m.modules() if isinstance(mod,(nn.Conv2d,nn.Linear))]; prune.global_unstructured([(mod,"weight") for mod in modules],pruning_method=prune.L1Unstructured,amount=sparsity)
  module_names={id(mod):name for name,mod in m.named_modules()}; masks={module_names[id(mod)]+".weight":mod.weight_mask.detach().clone() for mod in modules}
  for mod in modules: prune.remove(mod,"weight")
 else: raise ValueError(method)
 if method!="l1":
  with torch.no_grad():
   for n,p in m.named_parameters():
    if n in masks: p.mul_(masks[n].to(DEVICE))
 opt=torch.optim.SGD(m.parameters(),lr=.005,momentum=.9,weight_decay=5e-4); train(m,train_loader,opt,1 if MODE=="FAST" else 3,masks)
 return m
def adaptive_amfi(base,sparsity,seed):
 # equal budget: same calibration batches and one short fine-tune epoch per candidate; validation selects max accuracy, then freezes coefficients
 trials=[]
 for name,c in COEFFICIENTS.items():
  m=prune_method(base,"amfi",sparsity,seed,c); trials.append((eval_acc(m,val_loader),name,c,m))
 best=max(trials,key=lambda z:z[0]); return best[3],{"selected":best[1],"coefficients":best[2],"validation_accuracy":best[0],"candidate_compute":len(trials)}

In [ ]:
# Drive-backed checkpoints make rerunning cells resumable without retraining completed pruning conditions.
_original_baseline_model=baseline_model
def baseline_model(seed):
 path=ROOT/f"baseline_{MODE}_{seed}.pt"
 if path.exists():
  state=torch.load(path,map_location=DEVICE,weights_only=True); m=make_model().to(DEVICE); m.load_state_dict(state["model"]); print("Resumed baseline",seed); return m
 return _original_baseline_model(seed)
_original_prune_method=prune_method
def prune_method(base,method,sparsity,seed,coeff=(.25,.35,.25,.15)):
 tag=hashlib.sha1(json.dumps(list(coeff)).encode()).hexdigest()[:8]; path=ROOT/f"pruned_{MODE}_{EPOCHS}_{CALIBRATION_BATCHES}_{method}_{seed}_{int(sparsity*100)}_{tag}.pt"
 if path.exists():
  m=copy.deepcopy(base).to(DEVICE); m.load_state_dict(torch.load(path,map_location=DEVICE,weights_only=True)); print("Resumed pruning checkpoint",path.name); return m
 m=_original_prune_method(base,method,sparsity,seed,coeff); torch.save(m.state_dict(),path); return m

In [ ]:
def measured_sizes(model):
 dense=io.BytesIO(); torch.save(model.state_dict(),dense); dense_bytes=dense.tell()
 sparse_state={k:(v.detach().cpu().to_sparse() if v.is_floating_point() and torch.count_nonzero(v)<v.numel() else v.detach().cpu()) for k,v in model.state_dict().items()}
 sparse=io.BytesIO(); torch.save(sparse_state,sparse); return dense_bytes,sparse.tell()
def measured_latency(model,repeats=20):
 model.eval(); x=torch.randn(1,3,32,32,device=DEVICE)
 with torch.inference_mode():
  for _ in range(5): model(x)
  if DEVICE.type=="cuda": torch.cuda.synchronize()
  times=[]
  for _ in range(repeats):
   start=time.perf_counter(); model(x)
   if DEVICE.type=="cuda": torch.cuda.synchronize()
   times.append(time.perf_counter()-start)
 return float(np.mean(times)*1000),float(1/np.mean(times))
def measured_macs(model):
 count={"macs":0}; hooks=[]
 def conv(m,inputs,out): count["macs"]+=int(out.numel()*(m.in_channels//m.groups)*np.prod(m.kernel_size))
 def linear(m,inputs,out): count["macs"]+=int(out.numel()*m.in_features)
 for m in model.modules():
  if isinstance(m,nn.Conv2d): hooks.append(m.register_forward_hook(conv))
  elif isinstance(m,nn.Linear): hooks.append(m.register_forward_hook(linear))
 model.eval()
 with torch.inference_mode(): model(torch.zeros(1,3,32,32,device=DEVICE))
 for h in hooks: h.remove()
 return count["macs"]

## 21–23. Sparsity sweep, ablation, hyperparameter analysis

In [ ]:
rows=[]; ablations=[]; hypers=[]
ABLATIONS={"magnitude":(1,0,0,0),"mag_grad":(.5,.5,0,0),"mag_act":(.5,0,.5,0),"mag_layer":(.5,0,0,.5),"mag_grad_act":(1/3,1/3,1/3,0),"complete":(.25,.35,.25,.15)}
for seed in SEEDS:
 base=baseline_model(seed)
 for sparsity in SPARSITIES:
  for method in ("random","magnitude","l1","amfi"):
   m=prune_method(base,method,sparsity,seed); rows.append({"method":method,"target_sparsity":sparsity,"actual_sparsity":sum((p==0).sum().item() for n,p in m.named_parameters() if p.ndim>=2)/sum(p.numel() for p in m.parameters() if p.ndim>=2),"seed":seed,"validation_accuracy":eval_acc(m,val_loader),"parameter_count":sum(p.numel() for p in m.parameters()),"nonzero_parameters":sum(torch.count_nonzero(p).item() for p in m.parameters())})
  am,info=adaptive_amfi(base,sparsity,seed); rows.append({"method":"adaptive_amfi","target_sparsity":sparsity,"actual_sparsity":sparsity,"seed":seed,"validation_accuracy":eval_acc(am,val_loader),"selected_coefficients":json.dumps(info)})
  for name,c in ABLATIONS.items():
   mm=prune_method(base,"amfi",sparsity,seed,c); ablations.append({"variant":name,"sparsity":sparsity,"seed":seed,"validation_accuracy":eval_acc(mm,val_loader)})
  for name,c in COEFFICIENTS.items():
   mm=prune_method(base,"amfi",sparsity,seed,c); hypers.append({"candidate":name,"coefficients":json.dumps(c),"sparsity":sparsity,"seed":seed,"validation_accuracy":eval_acc(mm,val_loader)})
 pd.DataFrame(rows).to_csv(RESULTS,index=False); pd.DataFrame(ablations).to_csv(ROOT/"ablation_results.csv",index=False); pd.DataFrame(hypers).to_csv(ROOT/"hyperparameter_results.csv",index=False); print("Checkpointed rows",len(rows))

In [ ]:
# Measure checkpoint files, dense/sparse state serialization, MACs, and latency for each frozen ResNet condition.
source_dense_bytes_by_seed={}
for row in rows:
 target_method="amfi" if row["method"]=="adaptive_amfi" else row["method"]; coeff=(.25,.35,.25,.15)
 if row["method"]=="adaptive_amfi": coeff=tuple(json.loads(row["selected_coefficients"])["coefficients"])
 tag=hashlib.sha1(json.dumps(list(coeff)).encode()).hexdigest()[:8]; seed_id=int(row["seed"]); sparsity_id=int(row["target_sparsity"]*100); ckpt=ROOT/f"pruned_{MODE}_{EPOCHS}_{CALIBRATION_BATCHES}_{target_method}_{seed_id}_{sparsity_id}_{tag}.pt"
 measured=make_model().to(DEVICE); measured.load_state_dict(torch.load(ckpt,map_location=DEVICE,weights_only=True)); dense_bytes,sparse_bytes=measured_sizes(measured); latency_ms,throughput=measured_latency(measured); macs=measured_macs(measured)
 if seed_id not in source_dense_bytes_by_seed:
  baseline=torch.load(ROOT/f"baseline_{MODE}_{seed_id}.pt",map_location="cpu",weights_only=True); source_stream=io.BytesIO(); torch.save(baseline["model"],source_stream); source_dense_bytes_by_seed[seed_id]=source_stream.tell()
 source_dense_bytes=source_dense_bytes_by_seed[seed_id]
 row.update({"checkpoint_bytes":ckpt.stat().st_size,"dense_state_bytes":dense_bytes,"source_dense_state_bytes":source_dense_bytes,"sparse_serialized_bytes":sparse_bytes,"latency_ms":latency_ms,"throughput_samples_s":throughput,"macs":macs,"compression_ratio":source_dense_bytes/max(1,sparse_bytes),"logical_nonzero_ratio":sum(p.numel() for p in measured.parameters())/max(1,sum(torch.count_nonzero(p).item() for p in measured.parameters()))})
pd.DataFrame(rows).to_csv(RESULTS,index=False); print("Measured storage and runtime for",len(rows),"conditions")

## Structured channel baseline on a separately trained SmallCNN

In [ ]:
class NotebookSmallCNN(nn.Module):
 def __init__(self,width=16):
  super().__init__(); self.features=nn.Sequential(nn.Conv2d(3,width,3,padding=1),nn.ReLU(),nn.Conv2d(width,width*2,3,padding=1),nn.ReLU(),nn.AdaptiveAvgPool2d(1)); self.classifier=nn.Linear(width*2,10)
 def forward(self,x): return self.classifier(self.features(x).flatten(1))
def shrink_smallcnn(model,fraction):
 convs=[m for m in model.features if isinstance(m,nn.Conv2d)]; c1,c2=convs; fc=model.classifier
 def ids(score,n): return torch.argsort(score,descending=True,stable=True)[:max(1,round(n*(1-fraction)))].sort().values
 i1=ids(c1.weight.detach().abs().sum((1,2,3)),c1.out_channels); i2score=c2.weight.detach()[:,i1].abs().sum((1,2,3)); i2=ids(i2score,c2.out_channels)
 a=nn.Conv2d(c1.in_channels,len(i1),c1.kernel_size,padding=c1.padding).to(DEVICE); b=nn.Conv2d(len(i1),len(i2),c2.kernel_size,padding=c2.padding).to(DEVICE); f=nn.Linear(len(i2),10).to(DEVICE)
 with torch.no_grad(): a.weight.copy_(c1.weight[i1]); a.bias.copy_(c1.bias[i1]); b.weight.copy_(c2.weight[i2][:,i1]); b.bias.copy_(c2.bias[i2]); f.weight.copy_(fc.weight[:,i2]); f.bias.copy_(fc.bias)
 model.features=nn.Sequential(a,model.features[1],b,model.features[3],model.features[4]); model.classifier=f; return model
structured_rows=[]
for seed in SEEDS:
 seed_all(seed); base_path=ROOT/f"smallcnn_baseline_{MODE}_{seed}.pt"; base_small=NotebookSmallCNN().to(DEVICE)
 if base_path.exists(): base_small.load_state_dict(torch.load(base_path,map_location=DEVICE,weights_only=True))
 else:
  opt=torch.optim.SGD(base_small.parameters(),lr=.1,momentum=.9,weight_decay=5e-4); train(base_small,train_loader,opt,EPOCHS); torch.save(base_small.state_dict(),base_path)
 original=sum(p.numel() for p in base_small.parameters())
 for sp in SPARSITIES:
  out_path=ROOT/f"smallcnn_structured_{MODE}_{seed}_{int(sp*100)}.pt"; model=shrink_smallcnn(copy.deepcopy(base_small),sp)
  if out_path.exists(): model.load_state_dict(torch.load(out_path,map_location=DEVICE,weights_only=True))
  else:
   opt=torch.optim.SGD(model.parameters(),lr=.005,momentum=.9,weight_decay=5e-4); train(model,train_loader,opt,1 if MODE=="FAST" else 3); torch.save(model.state_dict(),out_path)
  count=sum(p.numel() for p in model.parameters()); dense_bytes,sparse_bytes=measured_sizes(model); source_dense_bytes,_=measured_sizes(base_small); latency_ms,throughput=measured_latency(model); macs=measured_macs(model)
  structured_rows.append({"method":"structured_smallcnn","seed":seed,"target_sparsity":sp,"actual_sparsity":1-count/original,"parameter_count":count,"validation_accuracy":eval_acc(model,val_loader),"checkpoint_bytes":out_path.stat().st_size,"dense_state_bytes":dense_bytes,"source_dense_state_bytes":source_dense_bytes,"sparse_serialized_bytes":sparse_bytes,"latency_ms":latency_ms,"throughput_samples_s":throughput,"macs":macs,"compression_ratio":source_dense_bytes/max(1,sparse_bytes),"logical_nonzero_ratio":count/max(1,sum(torch.count_nonzero(p).item() for p in model.parameters())),"nonzero_parameters":sum(torch.count_nonzero(p).item() for p in model.parameters())})
pd.DataFrame(structured_rows).to_csv(ROOT/"structured_results.csv",index=False)

## 24. Normalization comparison and runtime profile

In [ ]:
normalization_rows=[]
for seed in SEEDS:
 base=torch.load(ROOT/f"baseline_{MODE}_{seed}.pt",map_location=DEVICE,weights_only=True); bm=make_model().to(DEVICE); bm.load_state_dict(base["model"]); comp=importance(bm,cal_loader,CALIBRATION_BATCHES)
 for mode in ("layer_minmax","layer_mean","percentile"):
  for sparsity in (.5,.7,.8):
   mm=copy.deepcopy(bm); masks=make_masks(amfi_scores(comp,mode=mode),sparsity)
   with torch.no_grad():
    for n,p in mm.named_parameters():
     if n in masks: p.mul_(masks[n].to(DEVICE))
   normalization_rows.append({"seed":seed,"normalization":mode,"sparsity":sparsity,"validation_accuracy":eval_acc(mm,val_loader)})
pd.DataFrame(normalization_rows).to_csv(ROOT/"normalization_results.csv",index=False); print(pd.DataFrame(normalization_rows).groupby("normalization").validation_accuracy.agg(["mean","std"]))

In [ ]:
def measure_latency(model,repeats=50):
 model.eval(); x=torch.randn(1,3,32,32,device=DEVICE)
 with torch.inference_mode():
  for _ in range(10): model(x)
  if DEVICE.type=="cuda": torch.cuda.synchronize()
  ts=[]
  for _ in range(repeats):
   t=time.perf_counter(); model(x)
   if DEVICE.type=="cuda": torch.cuda.synchronize()
   ts.append(time.perf_counter()-t)
 return float(np.mean(ts)*1000),float(1/np.mean(ts))
profile_model=make_model().to(DEVICE); latency_ms,throughput=measure_latency(profile_model)
profile={"device":str(DEVICE),"gpu":torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,"baseline_dense_latency_ms":latency_ms,"baseline_throughput_samples_s":throughput,"parameter_count":sum(p.numel() for p in profile_model.parameters())}
torch.save(profile_model.state_dict(),ROOT/"profile_checkpoint.pt"); profile["checkpoint_bytes"]=(ROOT/"profile_checkpoint.pt").stat().st_size
(ROOT/"runtime_profile.json").write_text(json.dumps(profile,indent=2)); print(profile)

## 25. Freeze configurations, then final test evaluation

In [ ]:
# Test set is touched only after validation selection is frozen. Rebuild each selected condition from its seed baseline.
final=[]
for seed in SEEDS:
 state=torch.load(ROOT/f"baseline_{MODE}_{seed}.pt",map_location=DEVICE,weights_only=True); base=make_model().to(DEVICE); base.load_state_dict(state["model"]); final.append({"method":"baseline","seed":seed,"target_sparsity":0,"test_accuracy":eval_acc(base,test_loader)})
 for row in [r for r in rows if r["seed"]==seed]:
  if row["method"]=="adaptive_amfi":
   info=json.loads(row["selected_coefficients"]); model=prune_method(base,"amfi",row["target_sparsity"],seed,tuple(info["coefficients"]))
  else: model=prune_method(base,row["method"],row["target_sparsity"],seed)
  final.append({"method":row["method"],"seed":seed,"target_sparsity":row["target_sparsity"],"test_accuracy":eval_acc(model,test_loader)})
 small=NotebookSmallCNN().to(DEVICE); small.load_state_dict(torch.load(ROOT/f"smallcnn_baseline_{MODE}_{seed}.pt",map_location=DEVICE,weights_only=True))
 for sp in SPARSITIES:
  sm=shrink_smallcnn(copy.deepcopy(small),sp); sm.load_state_dict(torch.load(ROOT/f"smallcnn_structured_{MODE}_{seed}_{int(sp*100)}.pt",map_location=DEVICE,weights_only=True)); final.append({"method":"structured_smallcnn","seed":seed,"target_sparsity":sp,"test_accuracy":eval_acc(sm,test_loader)})
 testdf=pd.DataFrame(final); testdf.to_csv(ROOT/"final_test_results.csv",index=False); allrows=pd.DataFrame(rows).merge(testdf,on=["method","seed","target_sparsity"],how="left"); allrows["accuracy"]=allrows["test_accuracy"]; structs=pd.DataFrame(structured_rows).merge(testdf[testdf.method=="structured_smallcnn"],on=["method","seed","target_sparsity"],how="left"); structs["accuracy"]=structs["test_accuracy"]; combined=pd.concat([allrows,structs],ignore_index=True,sort=False); combined.to_csv(RESULTS,index=False); print(testdf)

## 26. Checkpoint size, measured latency and throughput

In [ ]:
# Include the frozen unpruned references in the shared result table for dashboard comparisons.
testdf=pd.read_csv(ROOT/"final_test_results.csv"); baseline_rows=testdf[testdf.method=="baseline"].copy(); baseline_rows["actual_sparsity"]=0.0; baseline_rows["accuracy"]=baseline_rows["test_accuracy"]; baseline_rows["validation_accuracy"]=np.nan
combined=pd.read_csv(RESULTS); combined=pd.concat([combined,baseline_rows],ignore_index=True,sort=False); combined.to_csv(RESULTS,index=False); print("Added baseline test rows:",len(baseline_rows))

In [ ]:
def measure_latency(model,repeats=50):
 model.eval(); x=torch.randn(1,3,32,32,device=DEVICE)
 with torch.inference_mode():
  for _ in range(10): model(x)
  if DEVICE.type=="cuda": torch.cuda.synchronize()
  ts=[]
  for _ in range(repeats):
   t=time.perf_counter(); model(x)
   if DEVICE.type=="cuda": torch.cuda.synchronize()
   ts.append(time.perf_counter()-t)
 return float(np.mean(ts)*1000),float(1/np.mean(ts))
# Dense zero-weight checkpoints generally do not shrink. Sparse tensor serialization must be measured separately; no FLOPs benefit is assumed for unstructured masks.

## 27–29. Visualizations, CSV exports, PDF report

In [ ]:
import matplotlib.pyplot as plt
df=pd.read_csv(RESULTS); fig,ax=plt.subplots(figsize=(8,5))
summary=df.groupby(["method","target_sparsity"])["accuracy"].agg(["mean","std"]).reset_index(); summary["std"]=summary["std"].fillna(0)
for name,g in summary.groupby("method"): ax.errorbar(g.target_sparsity,g["mean"],yerr=g["std"],marker="o",capsize=3,label=name)
ax.set(xlabel="Target sparsity",ylabel="Test accuracy after freeze",title="Measured accuracy by method (mean ± SD across seeds)"); ax.legend(); ax.grid(alpha=.2); fig.tight_layout(); fig.savefig(ROOT/"accuracy_vs_sparsity.png",dpi=160)
from reportlab.lib.pagesizes import letter
from reportlab.platypus import SimpleDocTemplate,Paragraph,Table,Spacer
from reportlab.lib.styles import getSampleStyleSheet
st=getSampleStyleSheet(); story=[Paragraph("AMFI Pruning: measured experiment report",st["Title"]),Spacer(1,12),Paragraph(f"Mode: {MODE}; seeds: {SEEDS}; device: {DEVICE}. Results are empirical notebook outputs.",st["BodyText"]),Spacer(1,12),Table([list(df.columns)]+df.astype(str).values.tolist(),repeatRows=1),Spacer(1,12),Paragraph("Accuracy by method (mean and standard deviation)",st["Heading2"]),Table([list(summary.columns)]+summary.astype(str).values.tolist(),repeatRows=1)]
SimpleDocTemplate(str(ROOT/"amfi_research_report.pdf"),pagesize=letter).build(story)
print("Saved CSV and PDF outputs in",ROOT)

In [ ]:
bundle=ROOT/"amfi_experiment_bundle"; checkpoint_bundle=bundle/"checkpoints"; checkpoint_bundle.mkdir(parents=True,exist_ok=True)
for seed in SEEDS:
 for src in [ROOT/f"baseline_{MODE}_{seed}.pt",ROOT/f"smallcnn_baseline_{MODE}_{seed}.pt"]:
  if src.exists(): shutil.copy2(src,checkpoint_bundle/src.name)
 for sp in SPARSITIES:
  src=ROOT/f"smallcnn_structured_{MODE}_{seed}_{int(sp*100)}.pt"
  if src.exists(): shutil.copy2(src,checkpoint_bundle/src.name)
print("Persisted baseline and structured checkpoints staged for bundle export")

In [ ]:
# Export mean and sample standard deviation by method and sparsity for the measured quality and resource metrics.
result_table=pd.read_csv(RESULTS); metrics=[c for c in ("accuracy","actual_sparsity","checkpoint_bytes","sparse_serialized_bytes","compression_ratio","latency_ms","throughput_samples_s","macs") if c in result_table.columns]
summary_stats=result_table.groupby(["method","target_sparsity"])[metrics].agg(["mean","std"]).reset_index(); summary_stats.columns=["_".join(str(part) for part in col if part) if isinstance(col,tuple) else str(col) for col in summary_stats.columns]; summary_stats.to_csv(ROOT/"statistics_summary.csv",index=False); print("Saved grouped mean/std statistics",summary_stats.shape)

In [ ]:
shutil.copy2(ROOT/"statistics_summary.csv",bundle/"statistics_summary.csv")

## 30. Experiment bundle export and resume point

In [ ]:
eid=str(uuid.uuid4()); bundle=ROOT/"amfi_experiment_bundle"; bundle.mkdir(exist_ok=True)
metadata={"experiment_id":eid,"created_at":pd.Timestamp.utcnow().isoformat(),"method":"multi_method_research","seed":SEEDS[0],"dataset":"CIFAR-10","architecture":"CIFAR ResNet-18 and SmallCNN","mode":MODE,"sparsity":max(SPARSITIES)}
(bundle/"experiment_metadata.json").write_text(json.dumps(metadata,indent=2))
for source,name in [(RESULTS,"experiment_results.csv"),(ROOT/"ablation_results.csv","ablation_results.csv"),(ROOT/"hyperparameter_results.csv","hyperparameter_results.csv"),(ROOT/"structured_results.csv","structured_results.csv"),(ROOT/"normalization_results.csv","normalization_results.csv"),(ROOT/"final_test_results.csv","final_test_results.csv"),(ROOT/"amfi_research_report.pdf","amfi_research_report.pdf"),(ROOT/"accuracy_vs_sparsity.png","accuracy_vs_sparsity.png")]: shutil.copy2(source,bundle/name)
files=[{"path":p.relative_to(bundle).as_posix(),"sha256":hashlib.sha256(p.read_bytes()).hexdigest()} for p in bundle.rglob("*") if p.is_file()]
(bundle/"manifest.json").write_text(json.dumps({"files":files},indent=2)); zip_path=ROOT/"amfi_experiment_bundle.zip"
with zipfile.ZipFile(zip_path,"w",zipfile.ZIP_DEFLATED) as z:
 for p in bundle.rglob("*"):
  if p.is_file(): z.write(p,p.relative_to(bundle))
print("Bundle ready:",zip_path); print("Resume: rerun cells; existing Drive CSVs are preserved as checkpoint artifacts.")